# 🧠 STRESS TRAJECTORY PREDICTION
## Simple & Easy-to-Understand ML Project

---

## 📌 Problem Definition
- **Goal:** Predict if someone is at risk of growing stress
- **Target:** `Stress_Risk` (Binary: "At Risk" vs "Not At Risk")
- **Primary Metric:** Recall (catch as many stressed individuals as possible)

### Why Recall?
Missing someone who IS stressed (False Negative) is worse than a false alarm.

In [ ]:
# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
from scipy.stats import chi2_contingency
import joblib
import os

print("✅ Libraries imported!")

In [ ]:
# ============================================
# STEP 2: LOAD DATA
# ============================================
df = pd.read_csv('Mental Health Dataset.csv')

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# ============================================
# STEP 3: UNDERSTAND THE DATA
# ============================================
print("📊 DATA TYPES:")
print(df.dtypes)

print("\n📊 MISSING VALUES:")
print(df.isnull().sum())

print(f"\n📊 DUPLICATES: {df.duplicated().sum():,}")

In [ ]:
# ============================================
# STEP 4: CHECK TARGET VARIABLE (Before Converting)
# ============================================
print("📊 ORIGINAL Growing_Stress Distribution:")
print(df['Growing_Stress'].value_counts())
print("\nPercentage:")
print((df['Growing_Stress'].value_counts(normalize=True) * 100).round(1))

In [ ]:
# ============================================
# STEP 5: CREATE BINARY TARGET VARIABLE
# Combine "Yes" and "Maybe" into "At Risk"
# ============================================

# Create new column: Stress_Risk
# - "At Risk" = Yes OR Maybe (they need attention)
# - "Not At Risk" = No

df['Stress_Risk'] = df['Growing_Stress'].apply(
    lambda x: 'At Risk' if x in ['Yes', 'Maybe'] else 'Not At Risk'
)

print("📊 NEW TARGET (Stress_Risk) Distribution:")
print(df['Stress_Risk'].value_counts())
print("\nPercentage:")
print((df['Stress_Risk'].value_counts(normalize=True) * 100).round(1))

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#2ecc71']  # Red for At Risk, Green for Not At Risk
df['Stress_Risk'].value_counts().plot(kind='bar', color=colors, ax=ax)
ax.set_title('Stress Risk Distribution (Binary)', fontsize=14, fontweight='bold')
ax.set_xlabel('Stress Risk')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
for i, v in enumerate(df['Stress_Risk'].value_counts()):
    ax.text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

# Check balance
ratio = df['Stress_Risk'].value_counts().max() / df['Stress_Risk'].value_counts().min()
print(f"\n✅ Imbalance Ratio: {ratio:.2f} - {'BALANCED' if ratio < 1.5 else 'IMBALANCED'}")

---
# 📊 EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# ============================================
# EDA 1: Stress Risk vs Mood Swings
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Cross-tabulation percentages
ct = pd.crosstab(df['Mood_Swings'], df['Stress_Risk'], normalize='index') * 100
print("Stress Risk by Mood Swings (%):\n", ct.round(1))

# Bar chart
ct.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Stress Risk by Mood Swings', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Stress Risk')
axes[0].tick_params(axis='x', rotation=0)

# Count plot
sns.countplot(data=df, x='Mood_Swings', hue='Stress_Risk', 
              palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Count by Mood Swings', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_mood_swings.png', dpi=150)
plt.show()

In [ ]:
# ============================================
# EDA 2: Stress Risk vs Family History
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ct = pd.crosstab(df['family_history'], df['Stress_Risk'], normalize='index') * 100
print("Stress Risk by Family History (%):\n", ct.round(1))

ct.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Stress Risk by Family History', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Stress Risk')
axes[0].tick_params(axis='x', rotation=0)

sns.countplot(data=df, x='family_history', hue='Stress_Risk', 
              palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Count by Family History', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_family_history.png', dpi=150)
plt.show()

In [ ]:
# ============================================
# EDA 3: Stress Risk vs Coping Struggles
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ct = pd.crosstab(df['Coping_Struggles'], df['Stress_Risk'], normalize='index') * 100
print("Stress Risk by Coping Struggles (%):\n", ct.round(1))

ct.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Stress Risk by Coping Struggles', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Stress Risk')
axes[0].tick_params(axis='x', rotation=0)

sns.countplot(data=df, x='Coping_Struggles', hue='Stress_Risk', 
              palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Count by Coping Struggles', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_coping_struggles.png', dpi=150)
plt.show()

In [ ]:
# ============================================
# EDA 4: Stress Risk vs Social Weakness
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ct = pd.crosstab(df['Social_Weakness'], df['Stress_Risk'], normalize='index') * 100
print("Stress Risk by Social Weakness (%):\n", ct.round(1))

ct.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Stress Risk by Social Weakness', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Stress Risk')
axes[0].tick_params(axis='x', rotation=0)

sns.countplot(data=df, x='Social_Weakness', hue='Stress_Risk', 
              palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Count by Social Weakness', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_social_weakness.png', dpi=150)
plt.show()

In [ ]:
# ============================================
# EDA 5: Stress Risk vs Days Indoors
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct = pd.crosstab(df['Days_Indoors'], df['Stress_Risk'], normalize='index') * 100
print("Stress Risk by Days Indoors (%):\n", ct.round(1))

ct.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Stress Risk by Days Indoors', fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Stress Risk')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(data=df, x='Days_Indoors', hue='Stress_Risk', 
              palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Count by Days Indoors', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_days_indoors.png', dpi=150)
plt.show()

In [ ]:
# ============================================
# EDA 6: CORRELATION HEATMAP
# ============================================

# Encode for correlation (temporary)
df_corr = df.drop(['Timestamp', 'Growing_Stress'], axis=1, errors='ignore').copy()
for col in df_corr.columns:
    if df_corr[col].dtype == 'object':
        df_corr[col] = LabelEncoder().fit_transform(df_corr[col].astype(str))

# Correlation matrix
corr = df_corr.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

# Correlations with target
print("\n📊 Correlations with Stress_Risk:")
print(corr['Stress_Risk'].drop('Stress_Risk').sort_values(ascending=False).round(3))

---
# 🧹 DATA CLEANING

In [ ]:
# ============================================
# DATA CLEANING
# ============================================

# Create clean copy
df_clean = df.copy()

# 1. Drop irrelevant columns
df_clean = df_clean.drop(['Timestamp', 'Growing_Stress', 'Country'], axis=1, errors='ignore')
print("✅ Dropped: Timestamp, Growing_Stress (kept Stress_Risk), Country")

# 2. Handle missing values (fill with mode)
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
print(f"✅ Missing values: {df_clean.isnull().sum().sum()}")

# 3. Remove duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"✅ Removed {before - len(df_clean):,} duplicates")

# 4. Standardize text (strip whitespace)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].str.strip()
print("✅ Text standardized")

print(f"\n📊 Clean dataset shape: {df_clean.shape}")
print(f"📊 Columns: {list(df_clean.columns)}")

---
# 🎯 FEATURE SELECTION

In [ ]:
# ============================================
# FEATURE SELECTION: CHI-SQUARE TEST
# ============================================

print("📊 Chi-Square Test Results (p-values):")
print("Features with p < 0.05 are significant predictors\n")

chi_results = {}
target = 'Stress_Risk'

for col in df_clean.columns:
    if col != target:
        contingency = pd.crosstab(df_clean[col], df_clean[target])
        chi2, p_val, dof, expected = chi2_contingency(contingency)
        chi_results[col] = p_val
        status = "✓ KEEP" if p_val < 0.05 else "✗ DROP"
        print(f"{col}: p = {p_val:.6f} → {status}")

# Keep significant features
significant_features = [f for f, p in chi_results.items() if p < 0.05]
print(f"\n✅ Significant features: {significant_features}")

In [ ]:
# ============================================
# FEATURE IMPORTANCE (DECISION TREE)
# ============================================

# Encode for preliminary analysis
df_temp = df_clean.copy()
for col in df_temp.columns:
    if df_temp[col].dtype == 'object':
        df_temp[col] = LabelEncoder().fit_transform(df_temp[col].astype(str))

X_temp = df_temp.drop('Stress_Risk', axis=1)
y_temp = df_temp['Stress_Risk']

# Train simple tree
dt_temp = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_temp.fit(X_temp, y_temp)

# Feature importance
importance = pd.DataFrame({
    'Feature': X_temp.columns,
    'Importance': dt_temp.feature_importances_
}).sort_values('Importance', ascending=True)

print("📊 Feature Importance:")
print(importance.sort_values('Importance', ascending=False))

# Plot
plt.figure(figsize=(10, 6))
plt.barh(importance['Feature'], importance['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importance (Decision Tree)', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

# Final selected features
selected_features = importance[importance['Importance'] > 0.01]['Feature'].tolist()
print(f"\n✅ Selected Features (importance > 0.01): {selected_features}")

---
# ⚙️ PREPROCESSING PIPELINE

In [ ]:
# ============================================
# PREPROCESSING PIPELINE
# ============================================

# Final dataset
df_final = df_clean.copy()

# Encode all categorical columns
encoders = {}
for col in df_final.columns:
    if df_final[col].dtype == 'object':
        encoders[col] = LabelEncoder()
        df_final[col] = encoders[col].fit_transform(df_final[col].astype(str))

print("✅ Encoded categories:")
for col, le in encoders.items():
    print(f"   {col}: {list(le.classes_)}")

# Split features and target
X = df_final.drop('Stress_Risk', axis=1)
y = df_final['Stress_Risk']

# Save feature column names
feature_columns = X.columns.tolist()
print(f"\n✅ Features: {feature_columns}")

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n✅ Train: {len(X_train):,} samples")
print(f"✅ Test: {len(X_test):,} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Features scaled")

# Class labels for later
class_labels = encoders['Stress_Risk'].classes_
print(f"\n📊 Class labels: {class_labels}")

---
# 🌳 MODEL 1: DECISION TREE

In [ ]:
# ============================================
# DECISION TREE - BASELINE
# ============================================

# Train model
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Predict
y_pred_dt = dt_model.predict(X_test)

# Metrics
print("📊 DECISION TREE - BASELINE RESULTS:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f} ⬅️ PRIMARY")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt):.4f}")

# Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
print(f"\nConfusion Matrix:\n{cm_dt}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=class_labels))

---
# 📈 MODEL 2: NAIVE BAYES

In [ ]:
# ============================================
# NAIVE BAYES - BASELINE
# ============================================

# Train model (uses scaled data)
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

# Predict
y_pred_nb = nb_model.predict(X_test_scaled)

# Metrics
print("📊 NAIVE BAYES - BASELINE RESULTS:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_nb):.4f} ⬅️ PRIMARY")
print(f"F1-Score:  {f1_score(y_test, y_pred_nb):.4f}")

# Confusion Matrix
cm_nb = confusion_matrix(y_test, y_pred_nb)
print(f"\nConfusion Matrix:\n{cm_nb}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, target_names=class_labels))

---
# 🔧 HYPERPARAMETER TUNING

In [ ]:
# ============================================
# TUNE DECISION TREE
# ============================================

dt_params = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params, cv=5, scoring='recall', n_jobs=-1
)
dt_grid.fit(X_train, y_train)

print("📊 TUNED DECISION TREE:")
print(f"Best Parameters: {dt_grid.best_params_}")
print(f"Best CV Recall: {dt_grid.best_score_:.4f}")

# Get tuned model
dt_tuned = dt_grid.best_estimator_
y_pred_dt_tuned = dt_tuned.predict(X_test)

print(f"\nTest Metrics:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt_tuned):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt_tuned):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt_tuned):.4f} ⬅️")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt_tuned):.4f}")

In [ ]:
# ============================================
# TUNE NAIVE BAYES
# ============================================

nb_params = {'var_smoothing': np.logspace(-12, -6, 20)}

nb_grid = GridSearchCV(
    GaussianNB(), nb_params, cv=5, scoring='recall', n_jobs=-1
)
nb_grid.fit(X_train_scaled, y_train)

print("📊 TUNED NAIVE BAYES:")
print(f"Best Parameters: {nb_grid.best_params_}")
print(f"Best CV Recall: {nb_grid.best_score_:.4f}")

# Get tuned model
nb_tuned = nb_grid.best_estimator_
y_pred_nb_tuned = nb_tuned.predict(X_test_scaled)

print(f"\nTest Metrics:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb_tuned):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb_tuned):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_nb_tuned):.4f} ⬅️")
print(f"F1-Score:  {f1_score(y_test, y_pred_nb_tuned):.4f}")

---
# 🏆 FINAL MODEL COMPARISON

In [ ]:
# ============================================
# FINAL COMPARISON
# ============================================

# Calculate final metrics
cm_dt_final = confusion_matrix(y_test, y_pred_dt_tuned)
cm_nb_final = confusion_matrix(y_test, y_pred_nb_tuned)

# Results table
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall ⬅️', 'F1-Score'],
    'Decision Tree': [
        f"{accuracy_score(y_test, y_pred_dt_tuned):.4f}",
        f"{precision_score(y_test, y_pred_dt_tuned):.4f}",
        f"{recall_score(y_test, y_pred_dt_tuned):.4f}",
        f"{f1_score(y_test, y_pred_dt_tuned):.4f}"
    ],
    'Naive Bayes': [
        f"{accuracy_score(y_test, y_pred_nb_tuned):.4f}",
        f"{precision_score(y_test, y_pred_nb_tuned):.4f}",
        f"{recall_score(y_test, y_pred_nb_tuned):.4f}",
        f"{f1_score(y_test, y_pred_nb_tuned):.4f}"
    ]
})

print("📊 FINAL MODEL COMPARISON:")
print(results.to_string(index=False))

# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_dt_final, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_labels, yticklabels=class_labels)
axes[0].set_title('Decision Tree (Tuned)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_nb_final, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=class_labels, yticklabels=class_labels)
axes[1].set_title('Naive Bayes (Tuned)', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices_comparison.png', dpi=150)
plt.show()

# Select best model
dt_recall = recall_score(y_test, y_pred_dt_tuned)
nb_recall = recall_score(y_test, y_pred_nb_tuned)

if dt_recall > nb_recall:
    best_model = "Decision Tree"
    print(f"\n🏆 SELECTED: Decision Tree (Higher Recall: {dt_recall:.4f} vs {nb_recall:.4f})")
else:
    best_model = "Naive Bayes"
    print(f"\n🏆 SELECTED: Naive Bayes (Higher Recall: {nb_recall:.4f} vs {dt_recall:.4f})")

In [ ]:
# ============================================
# INTERPRET CONFUSION MATRIX
# ============================================

print("📊 CONFUSION MATRIX INTERPRETATION:")
print("\nDecision Tree:")
tn, fp, fn, tp = cm_dt_final.ravel()
print(f"  True Negatives (TN):  {tn:,} - Correctly predicted 'Not At Risk'")
print(f"  False Positives (FP): {fp:,} - Incorrectly predicted 'At Risk'")
print(f"  False Negatives (FN): {fn:,} - MISSED 'At Risk' individuals ⚠️")
print(f"  True Positives (TP):  {tp:,} - Correctly predicted 'At Risk'")
print(f"  Recall = TP/(TP+FN) = {tp}/{tp+fn} = {tp/(tp+fn):.4f}")

print("\nNaive Bayes:")
tn, fp, fn, tp = cm_nb_final.ravel()
print(f"  True Negatives (TN):  {tn:,} - Correctly predicted 'Not At Risk'")
print(f"  False Positives (FP): {fp:,} - Incorrectly predicted 'At Risk'")
print(f"  False Negatives (FN): {fn:,} - MISSED 'At Risk' individuals ⚠️")
print(f"  True Positives (TP):  {tp:,} - Correctly predicted 'At Risk'")
print(f"  Recall = TP/(TP+FN) = {tp}/{tp+fn} = {tp/(tp+fn):.4f}")

---
# 💾 SAVE MODELS

In [ ]:
# ============================================
# SAVE ALL MODELS & OBJECTS
# ============================================

# Create models folder
os.makedirs('models', exist_ok=True)

# Save models
joblib.dump(dt_tuned, 'models/decision_tree_model.pkl')
joblib.dump(nb_tuned, 'models/naive_bayes_model.pkl')
print("✅ Saved: decision_tree_model.pkl")
print("✅ Saved: naive_bayes_model.pkl")

# Save scaler
joblib.dump(scaler, 'models/scaler.pkl')
print("✅ Saved: scaler.pkl")

# Save encoders
joblib.dump(encoders, 'models/encoders.pkl')
print("✅ Saved: encoders.pkl")

# Save feature columns
joblib.dump(feature_columns, 'models/feature_columns.pkl')
print("✅ Saved: feature_columns.pkl")

# Save results for Streamlit
results_dict = {
    'dt_accuracy': accuracy_score(y_test, y_pred_dt_tuned),
    'dt_precision': precision_score(y_test, y_pred_dt_tuned),
    'dt_recall': recall_score(y_test, y_pred_dt_tuned),
    'dt_f1': f1_score(y_test, y_pred_dt_tuned),
    'dt_cm': cm_dt_final.tolist(),
    'nb_accuracy': accuracy_score(y_test, y_pred_nb_tuned),
    'nb_precision': precision_score(y_test, y_pred_nb_tuned),
    'nb_recall': recall_score(y_test, y_pred_nb_tuned),
    'nb_f1': f1_score(y_test, y_pred_nb_tuned),
    'nb_cm': cm_nb_final.tolist(),
    'best_model': best_model,
    'class_labels': class_labels.tolist()
}
joblib.dump(results_dict, 'models/model_results.pkl')
print("✅ Saved: model_results.pkl")

print("\n🎉 All models and objects saved to 'models/' folder!")

---
# 🚀 DEPLOYMENT SUMMARY

## Files Created:
- `stress_prediction_simple.ipynb` - This notebook
- `app.py` - Streamlit dashboard
- `requirements.txt` - Dependencies
- `models/` - Saved models and objects

## To Run Streamlit:
```bash
python3 -m streamlit run app.py
```

## Key Changes:
1. **Binary Target:** Combined "Yes" + "Maybe" → "At Risk"
2. **Simplified Code:** Clear, step-by-step structure
3. **True Binary Classification:** Now uses 2 classes only